In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Install all packages from
!pip install -r ../requirements.txt

In [ ]:
import json, os, csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv("../stats/download_stats.csv")
df.head()

In [ ]:
from cernopendata_client.searcher import get_record_as_json, get_files_list, get_file_info_remote
from cernopendata_client.downloader import download_single_file, check_error
from cernopendata_client.verifier import get_file_info_local, verify_file_info

SERVER_HTTP_URI = "https://opendata.cern.ch"
protocol = "xrootd"

In [ ]:
recid = 85240
record_json = get_record_as_json(SERVER_HTTP_URI, recid, None, None)

In [ ]:
file_locations_info = get_files_list(SERVER_HTTP_URI, record_json, protocol, True)

In [ ]:
print(record_json['metadata']['files'][0]['uri'])
print(file_locations_info)

In [ ]:
with open('delphi_records_master.json', 'r') as cache_file:
    master_cache = json.load(cache_file)

In [ ]:
# Compare recids in remote master_cache with recids found in local download stats
master_recids = {int(r) for r in master_cache.keys()}
stats_recids = set(df['recid'].unique())
missing_in_stats = sorted(master_recids - stats_recids)
print(f"Recids in master_cache: {len(master_recids)}")
print(f"Unique recids in download stats: {len(stats_recids)}")
print(f"Recids in master_cache but missing in download stats: {len(missing_in_stats)}")
if missing_in_stats:
    print("Sample missing recids:", missing_in_stats[:10])

In [ ]:
# print metadata of missing recids from master_cache
for recid in missing_in_stats:  # print metadata for first 5 missing recids
    recid_str = str(recid)
    metadata = master_cache.get(recid_str)
    # print(f"Recid: {recid}")
    if metadata:
        if 'recid' not in metadata.keys():
            print(recid)
        # print(f'    Keys: {list(metadata.keys())}')
        # print(f'    Files: {len(metadata.get("files", []))}')
        # print()
    else:
        print("No metadata found in master_cache.")

In [ ]:
from pathlib import Path

def _extract_recids(payload):
    recids = set()
    if isinstance(payload, dict):
        for key in ('recid', 'id'):
            if key in payload:
                try:
                    recids.add(int(payload[key]))
                except (TypeError, ValueError):
                    pass
        for key in ('hits', 'records', 'datasets', 'results'):
            if isinstance(payload.get(key), list):
                for item in payload[key]:
                    recids |= _extract_recids(item)
    elif isinstance(payload, list):
        for item in payload:
            recids |= _extract_recids(item)
    elif isinstance(payload, (int, str)):
        try:
            recids.add(int(payload))
        except ValueError:
            pass
    return recids

def _load_list_recids(root_dir):
    aggregate = set()
    for path in sorted(Path(root_dir).glob('*')):
        if not path.is_file():
            continue
        raw = path.read_text(encoding='utf-8').strip()
        if not raw:
            continue
        try:
            aggregate |= _extract_recids(json.loads(raw))
            continue
        except json.JSONDecodeError:
            pass
        for line in raw.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                aggregate |= _extract_recids(json.loads(line))
                continue
            except json.JSONDecodeError:
                pass
            try:
                aggregate.add(int(line))
            except ValueError:
                continue
    return aggregate

lists_dir = Path('../lists')
list_recids = _load_list_recids(lists_dir) if lists_dir.exists() else set()
master_recids = {int(r) for r in master_cache.keys()}
missing_recids = sorted(master_recids - list_recids)

print(f"Master cache recids: {len(master_recids)}")
print(f"Recids found in lists: {len(list_recids)}")
print(f"Missing recids: {len(missing_recids)}")
if missing_recids:
    print("Sample missing recids:", missing_recids[:10])

In [ ]:
# detect sequences in master_cache recids
recid_list = sorted(int(recid) for recid in master_cache.keys())
sequences = []
start = recid_list[0]
end = recid_list[0]
for recid in recid_list[1:]:
    if recid == end + 1:
        end = recid
    else:
        sequences.append((start, end))
        start = recid
        end = recid
sequences.append((start, end))
print("Detected sequences of recids:")
for start, end in sequences:
    if start == end:
        print(f"{start}")
    else:
        print(f"{start} - {end}")

In [ ]:

# print average and max of sequences length
sequence_lengths = [end - start + 1 for start, end in sequences]
average_length = sum(sequence_lengths) / len(sequence_lengths)
max_length = max(sequence_lengths)
print(f"Average sequence length: {average_length}")
print(f"Max sequence length: {max_length}")

In [ ]:
from cernopendata_client.walker import get_list_directory
files = get_list_directory("/eos/opendata/delphi/", True, 60)
print(len(files))

In [ ]:
# attempt to download file from temp index 773
# 82852
# downloads/82852/Y00761.45.al      1      0
# downloads/82852/Y00792.41.al      1      0
# print(temp['files'][771]['remote'])
# download_single_file("downloads", temp['files'][773]['remote'], "http", "pycurl")
# download_single_file("downloads", "root://eospublic.cern.ch//eos/opendata/delphi/simulated-data/cern/gpym6143c1/v99_4/206.7/run20773.xsdst", "xrootd", "xrootd")

In [ ]:
# print one sample metadata file from delphi
import json
with open('../metadata/83701.json', 'r') as f:
    metadata = json.load(f)
    print(json.dumps(metadata, indent=2))

In [ ]:
# expand master cache using lists/ metadata
from pathlib import Path

MASTER_CACHE_PATH = Path('delphi_records_master.json')

def _iter_metadata_records(payload):
    if isinstance(payload, dict):
        yield payload
        for key in ('hits', 'records', 'datasets', 'results', 'entries', 'items'):
            seq = payload.get(key)
            if isinstance(seq, list):
                for item in seq:
                    yield from _iter_metadata_records(item)
    elif isinstance(payload, list):
        for item in payload:
            yield from _iter_metadata_records(item)

def _extract_recid_from_record(record):
    candidates = [
        record.get('recid'),
        record.get('id'),
        (record.get('metadata') or {}).get('recid'),
        (record.get('metadata') or {}).get('id'),
    ]
    for cand in candidates:
        if cand is None:
            continue
        try:
            return int(cand)
        except (TypeError, ValueError):
            continue
    return None

def _normalize_file_entry(raw):
    checksum = raw.get('checksum') or raw.get('checksum_value')
    checksum_type = raw.get('checksum_type')
    if isinstance(checksum, str) and ':' in checksum and not checksum_type:
        checksum_type, checksum = checksum.split(':', 1)
    remote = raw.get('uri') or raw.get('remote') or raw.get('url') or raw.get('link')
    size = raw.get('size') or raw.get('bytes') or raw.get('filesize')
    return {
        'label': raw.get('name') or raw.get('filename') or raw.get('path'),
        'remote': remote,
        'local_path': raw.get('local_path') or raw.get('local'),
        'size': size,
        'checksum_type': checksum_type,
        'checksum': checksum,
        'downloaded': bool(raw.get('downloaded', False)),
    }

def _extract_files(record):
    meta = record.get('metadata') or {}
    files = meta.get('files') or record.get('files') or []
    normalized = []
    for item in files:
        if isinstance(item, dict):
            normalized.append(_normalize_file_entry(item))
    return normalized

records_from_lists = {}
lists_dir = Path('../lists')
if lists_dir.exists():
    for path in sorted(lists_dir.glob('*')):
        if not path.is_file():
            continue
        raw_text = path.read_text(encoding='utf-8').strip()
        if not raw_text:
            continue
        decoded = None
        try:
            decoded = json.loads(raw_text)
        except json.JSONDecodeError:
            pass
        if decoded is None:
            for line in raw_text.splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    decoded = json.loads(line)
                except json.JSONDecodeError:
                    decoded = None
                if decoded is not None:
                    for rec in _iter_metadata_records(decoded):
                        recid = _extract_recid_from_record(rec)
                        if recid is not None and recid not in records_from_lists:
                            records_from_lists[recid] = rec
                    decoded = None
            continue
        for rec in _iter_metadata_records(decoded):
            recid = _extract_recid_from_record(rec)
            if recid is not None and recid not in records_from_lists:
                records_from_lists[recid] = rec

existing_recids = {int(r) for r in master_cache.keys()}
new_recids = sorted(set(records_from_lists.keys()) - existing_recids)

print(f"Parsed records from lists/: {len(records_from_lists)}")
print(f"Existing master-cache recids: {len(existing_recids)}")
print(f"New recids available: {len(new_recids)}")

if new_recids:
    for recid in new_recids:
        files = _extract_files(records_from_lists[recid])
        master_cache[str(recid)] = {
            'recid': recid,
            'done': False,
            'checked': False,
            'files': files,
        }
    with MASTER_CACHE_PATH.open('w', encoding='utf-8') as cache_file:
        json.dump(master_cache, cache_file, indent=2, sort_keys=True)
    print(f"Master cache expanded to {len(master_cache)} recids.")
else:
    print("Master cache already includes every recid found in lists/.")


In [ ]:
records_from_lists = {}
lists_dir = Path('../lists')
if lists_dir.exists():
    for path in sorted(lists_dir.glob('*')):
        if not path.is_file():
            continue
        raw_text = path.read_text(encoding='utf-8').strip()
        if not raw_text:
            continue
        decoded = None
        try:
            decoded = json.loads(raw_text)
        except json.JSONDecodeError:
            pass
        if decoded is None:
            for line in raw_text.splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    decoded = json.loads(line)
                except json.JSONDecodeError:
                    decoded = None
                if decoded is not None:
                    for rec in _iter_metadata_records(decoded):
                        recid = _extract_recid_from_record(rec)
                        if recid is not None and recid not in records_from_lists:
                            records_from_lists[recid] = rec
                    decoded = None
            continue
        for rec in _iter_metadata_records(decoded):
            recid = _extract_recid_from_record(rec)
            if recid is not None and recid not in records_from_lists:
                records_from_lists[recid] = rec

In [ ]:
# print random record from lists
import random
if records_from_lists:
    sample_recid = random.choice(list(records_from_lists.keys()))
    print(f"Sample recid from lists/: {sample_recid}")
    print(json.dumps(records_from_lists[sample_recid], indent=2))

In [ ]:
# print values of date_created, formats, number_events, number_files, size, type - secondary, collision_information - energy, categories - primary
for recid in list(records_from_lists.keys())[420:422]:
    metadata = records_from_lists[recid]
    # metadata = record.get('metadata', {})
    date_created = metadata.get('date_created')
    distributions = metadata.get('distribution', {})
    formats = distributions.get('formats')
    number_events = distributions.get('number_events')
    number_files = distributions.get('number_files')
    size = distributions.get('size')
    record_type = metadata.get('type')
    collision_info = metadata.get('collision_information', {})
    energy = collision_info.get('energy')
    categories = metadata.get('categories', {})
    primary_category = categories.get('primary')
    print(f"Recid: {recid}")
    print(f"  Date Created: {date_created}")    #['1994']
    print(f"  Formats: {formats}")  #['SHORT']
    print(f"  Number of Events: {number_events}")   #10500
    print(f"  Number of Files: {number_files}") #7
    print(f"  Size: {size}") #223534080
    print(f"  Type: {record_type}") #{'primary': 'Dataset', 'secondary': ['Simulated']}
    print(f"  Collision Energy: {energy}") #89-94 GeV
    print(f"  Primary Category: {primary_category}") #2 Fermion

In [ ]:
# import atlasopenmagic as atom
#
# atom.available_releases()
